# Investigation: Paneth -> Epithelial_Ki67+ rule (Multi-Stage Evaluation)

This notebook investigates the association rule **Paneth -> Epithelial_Ki67+** across different disease stages.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

# Add root to path for constants and utils
sys.path.append(os.path.abspath('../../'))
import constants

RADIUS = constants.CONFIG['RADIUS']
BANDWIDTH = constants.CONFIG.get('BANDWIDTH', RADIUS)
MIN_SUPPORT = constants.CONFIG['MIN_SUPPORT']
MIN_ABS_SUPPORT = constants.CONFIG.get('MIN_ABS_SUPPORT', 5)
MIN_CONFIDENCE = constants.CONFIG['MIN_CONFIDENCE']
MIN_LIFT = constants.CONFIG['MIN_LIFT']


## Step 1: Data Loading & Functional Subtypes

Load the cell and metadata tables, identify disease stages, and append the `functional_subtypes` array to every cell.

In [ ]:
def _add_functional_subtypes(df):
    if not constants.USE_FUNCTIONAL_MARKERS:
        return df
    functional_subtypes_list = [[] for _ in range(len(df))]
    idx_to_pos = {idx: i for i, idx in enumerate(df.index)}
    for base_type, markers in constants.CELLTYPE_MARKER_THRESHOLDS.items():
        type_mask = df["cell type"] == base_type
        for marker, threshold in markers.items():
            if marker in df.columns:
                mask = type_mask & (df[marker] > threshold)
                subtype_label = f"{base_type}_{marker}+"
                matching_indices = df.index[mask]
                for idx in matching_indices:
                    functional_subtypes_list[idx_to_pos[idx]].append(subtype_label)
    df["functional_subtypes"] = functional_subtypes_list
    return df

cell_table_path = '../../data/MIBIGutCsv/cell_table.csv'
fov_meta_path = '../../data/MIBIGutCsv/fovs_metadata.csv'
biopsy_meta_path = '../../data/MIBIGutCsv/biopsy_metadata.csv'

df_cells = pd.read_csv(cell_table_path)
df_fovs = pd.read_csv(fov_meta_path)
df_biopsy = pd.read_csv(biopsy_meta_path)

# Merge FOV with Biopsy to get Pathological score
df_fovs = pd.merge(df_fovs, df_biopsy[['Biopsy_ID', 'Pathological score']], left_on='Patient', right_on='Biopsy_ID', how='left')

# Define FOV lists for each cohort
control_cohorts = ['Duodenum_Cohort_Control_1', 'Duodenum_Cohort_Control_2', 'Colon_Cohort_Control']
control_fovs = df_fovs[df_fovs['Cohort'].isin(control_cohorts)]['FOV'].tolist()
mild_fovs = df_fovs[df_fovs['Pathological score'] == 'Mild']['FOV'].tolist()
severe_fovs = df_fovs[df_fovs['Pathological score'] == 'Severe']['FOV'].tolist()

# Normalize Coordinates
fov_to_size = df_fovs.set_index('FOV')['Size [um]'].to_dict()
def normalize_coords(row):
    size = fov_to_size.get(row['fov'], 800)
    res = 1024 if size == 400 else 2048
    return row['centroid_x'] * (size / res), row['centroid_y'] * (size / res)

df_cells[['x_um', 'y_um']] = df_cells.apply(lambda row: pd.Series(normalize_coords(row)), axis=1)

# Add Functional Subtypes
df_cells = _add_functional_subtypes(df_cells)

print(f"Total Cells: {len(df_cells)}")
print(f"Control FOVs: {len(control_fovs)}")
print(f"Mild FOVs: {len(mild_fovs)}")
print(f"Severe FOVs: {len(severe_fovs)}")


Total Cells: 713372
Control FOVs: 77
Mild FOVs: 143
Severe FOVs: 68


## Step 2: Evaluation Framework

Define a reusable function that iterates over a given list of FOVs, computes both binary and weighted metrics *per FOV*, and verifies how many FOVs pass the algorithm thresholds. Finally, it visualizes the FOV with the highest target cell density.

In [ ]:
def evaluate_cohort(cohort_name, fov_list, df_cells_all):
    print(f"==============================================")
    print(f"Evaluating Cohort: {cohort_name}")
    print(f"==============================================\n")
    
    binary_pass_count = 0
    weighted_pass_count = 0
    valid_fovs = 0
    
    global_binary_transactions = []
    global_weighted_transactions = []
    passing_binary_transactions = []
    passing_weighted_transactions = []
    
    max_target_count = -1
    max_target_fov = None
    
    for fov in fov_list:
        fov_data = df_cells_all[df_cells_all['fov'] == fov]
        if fov_data.empty: continue
        valid_fovs += 1
        
        coords = fov_data[['x_um', 'y_um']].values
        types = fov_data['cell type'].values
        subtypes = fov_data['functional_subtypes'].values
        
        # Track target cells for visualization (Paneth)
        target_count = sum(1 for t in types if t == 'Paneth')
        if target_count > max_target_count:
            max_target_count = target_count
            max_target_fov = fov
        
        nn = NearestNeighbors(radius=RADIUS).fit(coords)
        distances, neighbors_idx = nn.radius_neighbors(coords, return_distance=True)
        
        fov_binary_trans = []
        fov_weighted_trans = []
        
        for i, (idxs, dists) in enumerate(zip(neighbors_idx, distances)):
            if len(idxs) < constants.CONFIG.get("MIN_CELLS_PER_PATCH", 2):
                continue
            
            center_type = types[i]
            
            # Binary Transaction
            b_trans = set()
            b_trans.add(f"{center_type}_CENTER")
            for st in subtypes[i]: b_trans.add(f"{st}_CENTER")
            
            # Weighted Transaction
            w_trans = dict()
            w_trans[f"{center_type}_CENTER"] = 1.0
            for st in subtypes[i]: w_trans[f"{st}_CENTER"] = 1.0
                
            for n_idx, d in zip(idxs, dists):
                if n_idx != i:
                    ntype = types[n_idx]
                    b_trans.add(f"{ntype}_NEIGHBOR")
                    for st in subtypes[n_idx]: b_trans.add(f"{st}_NEIGHBOR")
                    
                    w = np.exp(-0.5 * (d / BANDWIDTH) ** 2)
                    w_trans[f"{ntype}_NEIGHBOR"] = w_trans.get(f"{ntype}_NEIGHBOR", 0.0) + w
                    for st in subtypes[n_idx]:
                        w_trans[f"{st}_NEIGHBOR"] = w_trans.get(f"{st}_NEIGHBOR", 0.0) + w
                        
            w_trans = {k: min(v, 1.0) for k, v in w_trans.items()}
            
            fov_binary_trans.append(b_trans)
            fov_weighted_trans.append(w_trans)
            global_binary_transactions.append(b_trans)
            global_weighted_transactions.append(w_trans)
        
        n_trans = len(fov_binary_trans)
        if n_trans == 0: 
            print(f"FOV: {fov} | 0 Transactions")
            continue
            
        # --- Binary Metrics Per FOV ---
        b_sup_ant = sum(1 for t in fov_binary_trans if 'Paneth_CENTER' in t) / n_trans
        b_sup_con = sum(1 for t in fov_binary_trans if 'Epithelial_Ki67+_NEIGHBOR' in t) / n_trans
        b_sup_joint = sum(1 for t in fov_binary_trans if 'Paneth_CENTER' in t and 'Epithelial_Ki67+_NEIGHBOR' in t) / n_trans
        b_conf = b_sup_joint / b_sup_ant if b_sup_ant > 0 else 0
        b_lift = b_conf / b_sup_con if b_sup_con > 0 else 0
        
        b_pass = False
        if b_sup_joint >= MIN_SUPPORT and b_conf >= MIN_CONFIDENCE and b_lift >= MIN_LIFT:
            binary_pass_count += 1
            b_pass = True
            
        # --- Weighted Metrics Per FOV ---
        w_sup_ant = sum(t.get('Paneth_CENTER', 0.0) for t in fov_weighted_trans) / n_trans
        w_sup_con = sum(t.get('Epithelial_Ki67+_NEIGHBOR', 0.0) for t in fov_weighted_trans) / n_trans
        w_sup_joint = sum(min(t.get('Paneth_CENTER', 0.0), t.get('Epithelial_Ki67+_NEIGHBOR', 0.0)) for t in fov_weighted_trans) / n_trans
        w_conf = w_sup_joint / w_sup_ant if w_sup_ant > 0 else 0
        w_lift = w_conf / w_sup_con if w_sup_con > 0 else 0
        
        w_pass = False
        w_min_sup = max(MIN_SUPPORT, MIN_ABS_SUPPORT / n_trans)
        if w_sup_joint >= w_min_sup and w_conf >= MIN_CONFIDENCE and w_lift >= MIN_LIFT:
            weighted_pass_count += 1
            w_pass = True
            
        if b_pass: passing_binary_transactions.extend(fov_binary_trans)
        if w_pass: passing_weighted_transactions.extend(fov_weighted_trans)
        
        print(f"FOV: {fov} | Transactions: {n_trans}")
        print(f"  [Binary]   Sup: {b_sup_joint:.4f} | Conf: {b_conf:.4f} | Lift: {b_lift:.4f} | PASS: {b_pass}")
        print(f"  [Weighted] Sup: {w_sup_joint:.4f} | Conf: {w_conf:.4f} | Lift: {w_lift:.4f} | PASS: {w_pass}\n")
            
    print(f"\n[Algorithm Comparison: Paneth_CENTER -> Epithelial_Ki67+_NEIGHBOR]")
    print(f"FOVs passing standard FP-Growth (Binary): {binary_pass_count} / {valid_fovs}")
    print(f"FOVs passing Weighted FP-Growth (Distance-decay): {weighted_pass_count} / {valid_fovs}")
    
    # --- Global Metrics Printout ---
    if not global_binary_transactions: return
    g_n_trans = len(global_binary_transactions)
    gb_sup_ant = sum(1 for t in global_binary_transactions if 'Paneth_CENTER' in t) / g_n_trans
    gb_sup_con = sum(1 for t in global_binary_transactions if 'Epithelial_Ki67+_NEIGHBOR' in t) / g_n_trans
    gb_sup_joint = sum(1 for t in global_binary_transactions if 'Paneth_CENTER' in t and 'Epithelial_Ki67+_NEIGHBOR' in t) / g_n_trans
    gb_conf = gb_sup_joint / gb_sup_ant if gb_sup_ant > 0 else 0
    gb_lift = gb_conf / gb_sup_con if gb_sup_con > 0 else 0
    
    gw_sup_ant = sum(t.get('Paneth_CENTER', 0.0) for t in global_weighted_transactions) / g_n_trans
    gw_sup_con = sum(t.get('Epithelial_Ki67+_NEIGHBOR', 0.0) for t in global_weighted_transactions) / g_n_trans
    gw_sup_joint = sum(min(t.get('Paneth_CENTER', 0.0), t.get('Epithelial_Ki67+_NEIGHBOR', 0.0)) for t in global_weighted_transactions) / g_n_trans
    gw_conf = gw_sup_joint / gw_sup_ant if gw_sup_ant > 0 else 0
    gw_lift = gw_conf / gw_sup_con if gw_sup_con > 0 else 0

    if passing_binary_transactions:
        p_n_trans = len(passing_binary_transactions)
        pb_sup_ant = sum(1 for t in passing_binary_transactions if 'Paneth_CENTER' in t) / p_n_trans
        pb_sup_con = sum(1 for t in passing_binary_transactions if 'Epithelial_Ki67+_NEIGHBOR' in t) / p_n_trans
        pb_sup_joint = sum(1 for t in passing_binary_transactions if 'Paneth_CENTER' in t and 'Epithelial_Ki67+_NEIGHBOR' in t) / p_n_trans
        pb_conf = pb_sup_joint / pb_sup_ant if pb_sup_ant > 0 else 0
        pb_lift = pb_conf / pb_sup_con if pb_sup_con > 0 else 0
        
        pw_sup_ant = sum(t.get('Paneth_CENTER', 0.0) for t in passing_weighted_transactions) / p_n_trans
        pw_sup_con = sum(t.get('Epithelial_Ki67+_NEIGHBOR', 0.0) for t in passing_weighted_transactions) / p_n_trans
        pw_sup_joint = sum(min(t.get('Paneth_CENTER', 0.0), t.get('Epithelial_Ki67+_NEIGHBOR', 0.0)) for t in passing_weighted_transactions) / p_n_trans
        pw_conf = pw_sup_joint / pw_sup_ant if pw_sup_ant > 0 else 0
        pw_lift = pw_conf / pw_sup_con if pw_sup_con > 0 else 0
    else:
        pb_sup_joint = pb_conf = pb_lift = 0
        pw_sup_joint = pw_conf = pw_lift = 0
    
    print(f"\n[Aggregate Metrics: ALL FOVs]")
    print(f"          {'Binary':>12} | {'Weighted':>12}")
    print(f"Approved: {binary_pass_count:>12} | {weighted_pass_count:>12}  (out of {valid_fovs} valid FOVs)")
    print(f"Support : {gb_sup_joint:>12.4f} | {gw_sup_joint:>12.4f}")
    print(f"Conf    : {gb_conf:>12.4f} | {gw_conf:>12.4f}")
    print(f"Lift    : {gb_lift:>12.4f} | {gw_lift:>12.4f}")

    print(f"\n[Aggregate Metrics: ONLY PASSING FOVs]")
    print(f"          {'Binary':>12} | {'Weighted':>12}")
    print(f"Support : {pb_sup_joint:>12.4f} | {pw_sup_joint:>12.4f}")
    print(f"Conf    : {pb_conf:>12.4f} | {pw_conf:>12.4f}")
    print(f"Lift    : {pb_lift:>12.4f} | {pw_lift:>12.4f}")
    
    if max_target_fov:
        print(f"\nVisualizing Representative FOV: {max_target_fov} ({max_target_count} target cells)")
        _plot_fov(max_target_fov, df_cells_all)

def _plot_fov(fov, df_cells_all):
    fov_df = df_cells_all[df_cells_all['fov'] == fov].copy()
    
    def get_plot_class(row):
        if row['cell type'] == 'Paneth': return 'Paneth'
        elif 'Epithelial_Ki67+' in row['functional_subtypes']: return 'Epithelial_Ki67+'
        elif row['cell type'] == 'Epithelial': return 'Epithelial_Other'
        return 'Other'
        
    fov_df['PlotClass'] = fov_df.apply(get_plot_class, axis=1)
    
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(10, 10))
    others = fov_df[fov_df['PlotClass'] == 'Other']
    sns.scatterplot(data=others, x='x_um', y='y_um', color='lightgrey', alpha=0.3, s=20)
    
    epi_other = fov_df[fov_df['PlotClass'] == 'Epithelial_Other']
    sns.scatterplot(data=epi_other, x='x_um', y='y_um', color='blue', alpha=0.5, s=20, label='Epithelial (Other)')
    
    epi_ki67 = fov_df[fov_df['PlotClass'] == 'Epithelial_Ki67+']
    sns.scatterplot(data=epi_ki67, x='x_um', y='y_um', color='green', s=60, edgecolor='black', zorder=8, label='Epithelial_Ki67+')
    
    target = fov_df[fov_df['PlotClass'] == 'Paneth']
    sns.scatterplot(data=target, x='x_um', y='y_um', color='red', s=70, edgecolor='black', zorder=10, label='Paneth')
    
    plt.title(f"Spatial Distribution in FOV {fov}")
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
    plt.show()


IndentationError: unexpected indent (1911659794.py, line 131)

## Step 3: Run Evaluation on Control Cohort

In [ ]:
evaluate_cohort("Control", control_fovs, df_cells)


## Step 4: Run Evaluation on Mild Cohort (Stage 1)

In [ ]:
evaluate_cohort("Mild", mild_fovs, df_cells)


## Step 5: Run Evaluation on Severe Cohort (Stage 3)

In [ ]:
evaluate_cohort("Severe", severe_fovs, df_cells)
